# Dynamic EGF kinase-phosphosite network optimization

This notebook shows how to run the upstream dynamic network optimization used to select an EGF-responsive signaling subnetwork.

It is intentionally self-contained: all data loading, graph construction, ILP formulation, optimization, validation, and plotting code is included here. It does not import helper functions from the repository `scripts/` directory.

The workflow is:

1. Load the EGF phosphoproteomics dynamic response table.
2. Keep the top dynamic phosphosites by absolute peak log2 fold change.
3. Load kinase-substrate interactions (the kinase-substrate pkl object can be obtained at: https://zenodo.org/records/18390833).
4. Build an alternating graph:
   `kinase -> phosphosite -> kinase -> phosphosite`.
5. Add an artificial `ROOT` connected to EGFR and an artificial `SINK` connected from transcription-factor phosphosites.
6. Solve a dynamic rooted prize-collecting Steiner-tree-style ILP.
7. Validate that the selected network is a DAG and respects temporal ordering.
8. Render the selected network with Graphviz.

This notebook covers only network selection. It does **not** include the downstream ODE/mechanistic model.


## 1. Package requirements

Required Python packages:

- `pandas`
- `numpy`
- `networkx`
- `cvxpy`
- `matplotlib`
- `pyarrow` or another parquet engine

A mixed-integer solver is required. The preferred solver is `GUROBI`; `GLPK_MI` can run smaller examples but is usually too slow for the full graph.

Graph rendering uses the Graphviz command-line tool `dot`. If Graphviz is not installed, the optimization still runs, but SVG/PNG rendering is skipped.


In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cvxpy as cp
import networkx as nx
import numpy as np
import pandas as pd

# Project code is imported as `from src.xxx import yyy`, so the repo root has to be on sys.path
# whether the notebook is run from its own folder or from the project root.
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dynamic_rpcst import (
    ALLOWED_ACTIVATION_TIMES,
    EGFR_UNIPROT,
    ROOT,
    SINK,
    TIME_COLORS,
    build_candidate_graph,
    default_solver,
    dynamic_rpcst_selection,
    find_project_root,
    load_kinsub,
    load_tf_protein_ids,
    prepare_peak_fc_table,
    prepare_solver_inputs,
    selected_with_artificial_terminals,
    temporal_violations,
    write_graphviz_dot,
    write_selected_outputs,
)

print("CVXPY installed solvers:", cp.installed_solvers())

## 2. Paths and user parameters

Adjust the values in this cell if file names or desired optimization settings change.

Important parameters:

- `TOP_FRACTION`: fraction of phosphosites retained by absolute peak log2 fold-change.
- `NODE_COST`: penalty for selecting each biological node. Higher values produce smaller subnetworks.
- `RESOURCE_REGEX`: optional filter for kinase-substrate resource names. For example, use `"Strict|Moderate"` to avoid lenient predictions.
- `TF_PROTEIN_LIST`: optional curated transcription-factor UniProt list. If omitted, TFs are inferred from protein descriptions.


In [ ]:
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Experiment/hme1_2/Data/Processed"
RESULTS_DIR = PROJECT_ROOT / "notebooks/07_dynamics_RPCST"
RESULTS_DIR.mkdir(exist_ok=True)

SOURCE_PARQUET = DATA_DIR / "20260819_hme1_2_EGF_dataset_Martin.parquet"
DYNAMIC_SITE_TABLE = DATA_DIR / "egf_peak_fc_by_site.parquet"
PKL_PATH = PROJECT_ROOT / "External_Data/Metadata/Martin"
KINSUB_PKL = PKL_PATH / "combined_kinsub_ce6e091017a229841db87586decbe46e.pkl"

# --- graph terminals and the time grid -------------------------------------------------
# These are copies of the defaults in src/dynamic_rpcst.py, restated here so every value the
# run depends on is visible and editable in one cell. They are passed explicitly at every call
# site below, so editing them here really does change the run.
ROOT = "ROOT"
SINK = "SINK"
EGFR_UNIPROT = "P00533"

# The grid a latent kinase activation time is drawn from. 0 lets a kinase sit upstream of the
# earliest measured site; the measured grid itself is {2, 5, 10, 15, 90}.
ALLOWED_ACTIVATION_TIMES = (0, 2, 5, 10, 15, 90)

TOP_FRACTION = 0.10
NODE_COST = 1.2
N_SELECTED_EDGES = None  # None lets the objective choose network size. Use an integer to force a size.
RESOURCE_REGEX = None    # Example: "Strict|Moderate"
TF_PROTEIN_LIST = None   # Optional Path to CSV/TSV/TXT curated TF UniProt IDs.
SOLVER = default_solver()   # GUROBI > GLPK_MI > SCIPY, whichever CVXPY can see
TIME_LIMIT_S = 180
MIP_GAP = 0.05

# --- figure styling --------------------------------------------------------------------
# Fill colour of a phosphosite node, keyed by its measured peak timepoint in minutes.
TIME_COLORS = {
    2: "#2563eb",
    5: "#16a34a",
    10: "#f59e0b",
    15: "#dc2626",
    90: "#7c3aed",
}

OUTPUT_PREFIX = RESULTS_DIR / f"collaborator_dynamic_rpcst_cost_{str(NODE_COST).replace('.', 'p')}"

print("Project root:", PROJECT_ROOT)
print("Using solver:", SOLVER)
print("Output prefix:", OUTPUT_PREFIX)

## 3. Prepare the dynamic phosphosite table

The optimization uses one row per localized phosphosite, with:

- a phosphosite identifier formatted as `UniProt_site`, matching the kinase-substrate table;
- absolute peak log2 fold-change versus starved/basal condition;
- peak response time in minutes.

If `data/egf_peak_fc_by_site.parquet` already exists, the notebook loads it. Otherwise it creates it from the original EGF parquet file.


In [ ]:
EGF_TIMES = ("2", "5", "10", "15", "90")
FC_COLS = [f"WT_log2:FC_EGF_{t}" for t in EGF_TIMES]

if DYNAMIC_SITE_TABLE.exists():
    dynamic_sites = pd.read_parquet(DYNAMIC_SITE_TABLE)
    if "kinsub_site_id" not in dynamic_sites.columns:
        dynamic_sites = dynamic_sites.copy()
        dynamic_sites["kinsub_site_id"] = dynamic_sites["protein_Id"].astype(str) + "_" + dynamic_sites["PhosSites"].astype(str)
else:
    dynamic_sites = prepare_peak_fc_table(SOURCE_PARQUET)
    dynamic_sites.to_parquet(DYNAMIC_SITE_TABLE, index=False)

print("Dynamic phosphosite table:", dynamic_sites.shape)
display(dynamic_sites.head())

## 4. Build the kinase-phosphosite candidate graph

The kinase-substrate table has:

- `source`: kinase UniProt accession;
- `target`: phosphosite identifier in `UniProt_site` format;
- `score`: interaction confidence;
- `resource`: source/prediction resource.

Graph construction rules:

- `kinase -> phosphosite` edges come from the kinase-substrate dataframe.
- `phosphosite -> kinase` edges are added when the phosphosite belongs to a kinase protein, allowing signal to continue through that kinase.
- `ROOT -> EGFR` anchors the network at the EGF receptor.
- TF phosphosites connect to `SINK`, defining acceptable output nodes.
- Only top dynamic phosphosites are retained as phosphosite nodes.
- The graph is pruned to nodes reachable from `ROOT`.


In [ ]:
kinsub = load_kinsub(KINSUB_PKL, RESOURCE_REGEX)
tf_protein_ids, tf_table, tf_source = load_tf_protein_ids(SOURCE_PARQUET, TF_PROTEIN_LIST)
candidate_graph, graph_stats = build_candidate_graph(kinsub,
                                                     dynamic_sites,
                                                     tf_protein_ids,
                                                     top_fraction=TOP_FRACTION,
                                                     root_kinase=EGFR_UNIPROT,
                                                     root_name=ROOT,
                                                     sink_name=SINK,)

tf_table.to_csv(OUTPUT_PREFIX.with_suffix(".tf_proteins.csv"), index=False)

print("Kinase-substrate rows:", len(kinsub))
print("TF source:", tf_source, "n=", len(tf_protein_ids))
print(json.dumps(graph_stats, indent=2))
print("Node types:")
print(pd.Series(nx.get_node_attributes(candidate_graph, "node_type")).value_counts())

## 5. ILP formulation and constraints

The optimization chooses a temporally coherent directed subnetwork connecting `ROOT` to `SINK`.

### Decision variables

For each candidate node $i$:

$$
x_i \in \{0, 1\}
$$

`x_i = 1` means node `i` is selected.

For each candidate edge $(u, v)$:

$$
y_{uv} \in \{0, 1\}
$$

`y_uv = 1` means edge `u -> v` is selected.

For each node $i$:

$$
d_i \in \mathbb{Z}_{\ge 0}
$$

`d_i` is a graph-distance/order variable. It forces selected edges to point away from `ROOT` and prevents cycles.

For each unmeasured kinase $k$:

$$
t_k \in \{0, 2, 5, 10, 15, 90\}
$$

This is the assigned latent activation time. It is selected from the observed experimental time grid.

### C1. Edge-to-node coupling

An edge can be selected only if both endpoint nodes are selected:

$$
y_{uv} \le x_u, \quad y_{uv} \le x_v
$$

This prevents edges floating outside the selected node set.

### C2. Rooted incoming support

Every selected non-root node must have at least one selected incoming edge:

$$
x_v \le \sum_{u:(u,v) \in E} y_{uv}
$$

This ensures every selected node is supported by upstream signal.

### C3. Sink-directed outgoing continuation

Every selected non-sink node must have at least one selected outgoing edge:

$$
x_u \le \sum_{v:(u,v) \in E} y_{uv}
$$

This keeps selected nodes on paths that can continue toward outputs.

### C4. Rooted DAG / cycle-breaking constraint

For every selected edge, the target must be farther from `ROOT` than the source:

$$
d_v \ge d_u + 1 \quad \text{if } y_{uv}=1
$$

Implemented with a big-M relaxation:

$$
d_v \ge d_u + 1 - M(1-y_{uv})
$$

This makes the selected subnetwork a directed acyclic graph.

### C5. Temporal ordering constraint

For measured phosphosites, time is fixed at the time point of maximum absolute fold-change. For unmeasured kinases, time is a latent variable selected from the experimental time grid.

For every selected biological edge:

$$
time(u) \le time(v)
$$

This prevents biologically inconsistent propagation such as:

```text
late phosphosite -> kinase -> early phosphosite
```

### C6. Optional edge-count constraint

If `N_SELECTED_EDGES` is set to an integer, exactly that many biological edges are selected. If it is `None`, the objective determines the size.

### Objective

The model minimizes:

$$
\text{missed prize} + \text{edge cost} + \text{node cost} + \text{tiny latent-time tie-break}
$$

Where:

- missed prize penalizes not selecting high-fold-change phosphosites;
- edge cost is `1 - interaction_score`;
- node cost controls sparsity;
- the tiny latent-time penalty chooses earlier feasible kinase times when otherwise tied.


## 6. The dynamic RPCST optimizer

The ILP described above is implemented by `dynamic_rpcst_selection()` in
**`src/dynamic_rpcst.py`**, imported in section 1. Its docstring documents every argument,
including the sharp edges: which nodes get a latent activation time, what happens to edges that
already violate the temporal order between two *known* times (they are deleted before the ILP is
built, not left for the solver), and what the `return_problem=True` dict does and does not
contain — note that its `nodes` and `edges` keys are the **candidate** lists, not the selected
ones.

## 7. Run the optimization

The code below removes the explicit `ROOT`/`SINK` from the candidate graph and lets the ILP add artificial terminal edges internally. This keeps biological edges and artificial connectivity edges separate.


In [ ]:
solver_graph, prizes, receptors, tf_sites, latent_time_nodes, activation_times = prepare_solver_inputs(
    candidate_graph,
    root_kinase=EGFR_UNIPROT,
    root_name=ROOT,
    sink_name=SINK,
)

print("Solver graph nodes:", solver_graph.number_of_nodes())
print("Solver graph edges:", solver_graph.number_of_edges())
print("Prized phosphosites:", len(prizes))
print("TF output phosphosites:", len(tf_sites))
print("Latent-time kinase nodes:", len(latent_time_nodes))

start = time.perf_counter()
result = dynamic_rpcst_selection(
    solver_graph,
    prizes=prizes,
    activation_times=activation_times,
    receptors=receptors,
    tfs=tf_sites,
    root_name=ROOT,
    sink_name=SINK,
    receptor_kinase=EGFR_UNIPROT,
    n_edges=N_SELECTED_EDGES,
    node_penalty=NODE_COST,
    latent_time_nodes=latent_time_nodes,
    latent_time_allowed_values=ALLOWED_ACTIVATION_TIMES,
    latent_time_penalty=1e-6,
    solver=SOLVER,
    mip=MIP_GAP,
    time_limit=TIME_LIMIT_S,
    include_artificial_terminals=False,
    return_problem=True,
)
solve_time = time.perf_counter() - start
selected_graph = result["subgraph"]

print("Status:", result["status"])
print("Objective:", result["objective_value"])
print("Solve time, seconds:", round(solve_time, 1))
print("Selected biological nodes:", selected_graph.number_of_nodes())
print("Selected biological edges:", selected_graph.number_of_edges())
print("Selected node types:")
print(pd.Series(nx.get_node_attributes(selected_graph, "node_type")).value_counts())

## 8. Validate the selected network

These checks are useful to share with collaborators because they confirm the selected network obeys the intended biological and graph constraints.


In [ ]:
selected_full = selected_with_artificial_terminals(result)
violations = temporal_violations(selected_graph)
summary = {
    "node_cost": NODE_COST,
    "status": result["status"],
    "objective_value": result["objective_value"],
    "solve_time_s": solve_time,
    "n_selected_nodes": selected_graph.number_of_nodes(),
    "n_selected_edges": selected_graph.number_of_edges(),
    "n_selected_kinases": sum(1 for _, d in selected_graph.nodes(data=True) if d.get("node_type") == "kinase"),
    "n_selected_phosphosites": sum(1 for _, d in selected_graph.nodes(data=True) if d.get("node_type") == "phosphosite"),
    "is_dag": nx.is_directed_acyclic_graph(selected_graph),
    "has_root_to_sink_path_with_artificial_terminals": nx.has_path(selected_full, ROOT, SINK),
    "n_temporal_violations": len(violations),
    "temporal_violations": violations[:20],
    "latent_time_values": result["latent_time_values"],
}
print(json.dumps(summary, indent=2))

assert summary["is_dag"]
assert summary["has_root_to_sink_path_with_artificial_terminals"]
assert summary["n_temporal_violations"] == 0

## 9. Save outputs

Outputs are written to `results/`:

- selected network as pickle;
- selected node table;
- selected edge table;
- summary JSON;
- TF protein list used for sink construction.


In [ ]:
output_paths = write_selected_outputs(selected_graph, summary, OUTPUT_PREFIX)
print(json.dumps(output_paths, indent=2))

## 10. Render the selected network with Graphviz

This cell creates a DOT file and, if the `dot` command is available, renders SVG and PNG figures.

Legend:

- blue boxes: kinase nodes with assigned latent activation time;
- colored ellipses: phosphosite nodes, colored by measured peak activation time;
- gray solid arrows: kinase-to-phosphosite interactions;
- orange dashed arrows: phosphosite-to-kinase propagation edges.


In [ ]:
dot_path = OUTPUT_PREFIX.with_suffix(".dot")
svg_path = OUTPUT_PREFIX.with_suffix(".svg")
png_path = OUTPUT_PREFIX.with_suffix(".png")
write_graphviz_dot(selected_graph,
                   dot_path,
                   f"Dynamic EGF RPCST selected network, node cost={NODE_COST}",
                   metadata_path=SOURCE_PARQUET,
                   time_colors=TIME_COLORS,)

if shutil.which("dot"):
    subprocess.run(["dot", "-Tsvg", str(dot_path), "-o", str(svg_path)], check=True)
    subprocess.run(["dot", "-Tpng", str(dot_path), "-o", str(png_path)], check=True)
    print("Wrote:", svg_path)
    print("Wrote:", png_path)
else:
    print("Graphviz 'dot' not found. Wrote DOT only:", dot_path)

print("DOT:", dot_path)

## 11. Practical notes

- For the full top-10% graph, GUROBI is strongly recommended.
- If optimization is slow, try increasing `NODE_COST`, filtering resources with `RESOURCE_REGEX`, or reducing `TOP_FRACTION`.
- The current TF set is inferred from protein descriptions unless a curated list is provided through `TF_PROTEIN_LIST`.
- Selected kinase activation times are latent variables chosen from the experimental grid. They should be interpreted as model-assigned times, not direct measurements.
- The selected network is a temporally constrained DAG, intended for interpretable signal-flow reconstruction from EGFR to TF phosphosites.


## 12. Observations

Code and method review, 2026-09-15. Everything below is **recorded, not yet acted on** — the
notebook and `src/dynamic_rpcst.py` still behave exactly as they did when the saved
`collaborator_dynamic_rpcst_cost_1p2.*` outputs were produced. The numbers quoted are derived
from that run's `summary.json` together with its `selected_nodes.csv` / `selected_edges.csv`,
except where a claim is marked as verified on synthetic data.

### 12.1 The prize term is ~98% a constant

Constraint **C3** ("every selected non-sink node needs an outgoing selected edge") together with
C1 and C4 means every selected node must lie on a complete `ROOT` → `SINK` path. In the candidate
graph a phosphosite only gets an out-edge if its protein is a kinase (→ kinase node) or a
transcription factor (→ `SINK`). **Every other phosphosite has out-degree 0 and can never be
selected, no matter how large its prize.** Verified by running `build_candidate_graph` on a
controlled synthetic instance: the two highest-prize sites came out with out-degree 0 and were
structurally excluded from every feasible solution.

Decomposing the saved objective:

| term | value |
|---|---|
| node cost (1.2 × 33 selected nodes) | 39.60 |
| edge cost Σ(1 − w) over 37 selected edges | 3.46 |
| latent-time tie-break (1e-6 × Σt) | 0.0001 |
| **missed prize (residual)** | **≈ 3010.5** |
| collected prize (24 selected sites) | 53.28 |
| **objective_value** | **3053.59** |

So **1.7% of the prize on offer was collected**, and the objective is dominated by a constant
that no solution could ever reduce. A corroborating sign: the selected sites average
|log2FC| = **2.22**, well below the candidate pool's implied ≈ 3.5 — the optimiser is not choosing
the biggest prizes, it is choosing whatever connectivity permits. This is not really
prize-collecting Steiner-tree behaviour; `NODE_COST` and the edge costs are doing nearly all of
the work.

Worth deciding deliberately: either relax C3 for non-TF leaves (classic PCST lets a path stop
anywhere), or stop describing the result as prize-driven.

### 12.2 Bugs

**`prepare_peak_fc_table` crashes on any site with no peak time.** `WT_peak:FC_EGF` is NaN when
the profile is entirely missing (documented in `CLAUDE.md` → *Peak-timing columns*).
`.astype(str)` turns that into the string `"nan"`, after which `astype("Int64")` raises
`ValueError: invalid literal for int() with base 10: 'nan'` and the fold-change lookup raises
`KeyError: 'WT_log2:FC_EGF_nan'`. Both reproduced. The same happens for a `full` / `starve` peak
label. It survives today only because the cached `egf_peak_fc_by_site.parquet` already exists —
and that cache is **never invalidated against its source**, so a changed source parquet is
silently ignored.

**Solver status is never checked.** `MIP_GAP = 0.05` with `TIME_LIMIT_S = 180`, and the saved run
took **140 s** — close enough to the wall that the next run may hit it. `status == "optimal"`
here means *"within a 5% optimality gap"*, not proven optimal, and the achieved gap is never
recorded. If the solver fails outright, `edge_vars.value` is `None` → empty selection (the
`has_root_to_sink_path` assertion does catch that, so it fails loudly — but silent
5%-suboptimality does not).

**GUROBI solver arguments are probably wrong.** `solve_kwargs.update({"time_limit": ...,
"mipGap": ...})` — Gurobi's parameter is `TimeLimit`. `mipGap` resolves case-insensitively;
`time_limit` does not. Not testable in the current environment (no Gurobi installed), but worth
checking before the collaborator runs it.

**86 latent times are reported, 9 kinases are selected.** The activation time of an *unselected*
kinase is still a free variable that enters the objective, so the tie-break pins it to `-0.0`.
`summary.json` therefore publishes activation times for 77 kinases that are not in the network,
which is genuinely misleading to a reader. It also costs 86 × 6 = 516 unnecessary binary
variables.

**Minor.** `FC_COLS` is dead code. `Path.with_suffix` on `OUTPUT_PREFIX` would truncate the name
if `NODE_COST` ever produced a string containing a dot. The `return_problem=True` dict's `nodes`
and `edges` keys are the **candidate** lists, not the selected ones (now documented in the
docstring, but easy to misread).

### 12.3 The validation doesn't validate

All three assertions in section 8 re-check constraints the ILP already enforced, using the ILP's
own variable values:

| assertion | enforced by |
|---|---|
| `is_dag` | C4 |
| `has_root_to_sink_path_with_artificial_terminals` | C2 + C3 + C4 |
| `n_temporal_violations == 0` | C5 |

They are fine as regression guards against a solver bug, but they are **not evidence about the
biology**, and the section heading ("confirm the selected network obeys the intended biological
and graph constraints") reads as though they are.

What is missing: an assertion on `result["status"]` and the achieved MIP gap, and any stability
check at all. How much does the selected network change with `NODE_COST` at 1.0 / 1.2 / 1.5,
`TOP_FRACTION` at 0.05 / 0.10, or a different solver? Given that ~98% of the objective is a
constant (12.1), the selection is likely to be quite unstable — and that is measurable.

### 12.4 Approach-level concerns

**Sign is discarded.** The prize is `peak_abs_log2_fc_vs_starve` and the
`phosphosite → kinase` edge is added whenever the modified protein is a kinase, regardless of
direction. A **dephosphorylated** site is rewarded exactly like an activating one, and is allowed
to switch its kinase on. Known inhibitory sites (RAF1 S259, GSK3B S9, …) therefore propagate
activation.

**Peak time is a noisy argmax, and this run shows it.** `MAPK3_Y204` is assigned t = 5 min while
`MAPK3_T202` is assigned t = 90 min. Those are the two residues of the ERK1 TEY activation loop —
they are phosphorylated together. The 85-minute disagreement then forces `P27361` (ERK1) to a
latent activation time of **90 min**, which is biologically wrong: ERK1 activates within 2–5 min.
Every kinase time in the model inherits this instability.

**The top-10% cut is amplitude-only, on unnormalised data.** No responsiveness filter is applied,
and `CLAUDE.md` → *Known issues* records a median `log2:FC` of **+0.566 across all 8723 sites at
EGF 5 min** caused by missing sample-loading normalisation. The candidate set is therefore ranked
partly on that shift. `filter_by_ffdr()` in `src/filters.py` (limma omnibus F, BH-FDR) is the
existing tool for this.

**`allow_equal_time=True` makes C5 close to vacuous.** The selected sites' peak times were
1 × 2 min, 10 × 5 min, 11 × 10 min, 2 × 90 min — nothing at 15. With at most 6 levels and equality
permitted, 21 of 24 sites sit in two bins, so most edges satisfy the temporal constraint
trivially.

**The transcription-factor set is a regex over UniProt description text**
(`"transcription factor|transcription regulator|DNA-binding protein"`), and it defines every
terminal the optimisation may end on — which, given 12.1, determines what can be selected at all.
That is a great deal of weight on a text heuristic. `TF_PROTEIN_LIST` already exists for supplying
a curated list instead.

**Formulation.** C4 uses a big-M of |V| (an MTZ-style ordering constraint), which gives a weak LP
relaxation and is the likely reason for the 140 s solve. A single-commodity flow formulation would
be substantially tighter on a graph this size.